# UrbanPulse - Bangalore Traffic & AQI Correlation Analysis

This notebook conducts a rigorous correlation analysis to evaluate the interactions between traffic speed, vehicle congestion levels, regional weather parameters, and Air Quality Indexes (AQI) across Bangalore.

## Analysis Tasks
1. **Load Master Dataset**: Imports the consolidated `data/processed/master_df.csv` file.
2. **Correlation Heatmap**: Visualizes the Pearson correlation of all numeric features with respect to the `AQI` target column.
3. **Fine Dust & Speed Interactions**: Creates a scatter plot comparing `avg_speed_kmph` against `PM2.5` concentrations, color-coded by weekend indicators (`is_weekend`).
4. **Station-by-Station Pearson r**: Calculates the Pearson correlation coefficient between traffic `congestion_level` and `AQI` for each localized monitoring zone.
5. **Insights Report**: Generates and compiles the top 3 analytics insights to `assets/insights.md`.

In [1]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Configure aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print("Libraries successfully loaded.")

### Step 1: Load Consolidated Master Dataset

In [2]:
master_path = "../data/processed/master_df.csv"
df = pd.read_csv(master_path)
print(f"Master dataset loaded successfully. Shape: {df.shape[0]} rows, {df.shape[1]} columns.")

### Step 2: Correlation Heatmap (All Numeric Features vs AQI)
We isolate all numeric variables, drop geographical coordinates (`latitude`, `longitude`), and compute the Pearson correlation matrix to plot a localized heatmap focused on `AQI` correlations.

In [3]:
# Select numeric features and filter coordinates
numeric_cols = df.select_dtypes(include=[np.number]).columns
numeric_cols = [c for c in numeric_cols if c not in ['latitude', 'longitude']]

# Compute correlation matrix
corr_matrix = df[numeric_cols].corr()

# Sort features by correlation with AQI
aqi_corr_sorted = corr_matrix[['AQI']].sort_values(by='AQI', ascending=False)

# Plot heatmap
plt.figure(figsize=(6, 8))
sns.heatmap(aqi_corr_sorted, annot=True, cmap="coolwarm", fmt=".3f", cbar=True, vmin=-1, vmax=1)
plt.title("Pearson Correlation: Features vs. AQI Target", fontsize=13, fontweight='bold', pad=15)
plt.ylabel("Features", fontsize=11)
plt.tight_layout()
plt.show()

### Step 3: Vehicle Speed vs. PM2.5 (Colored by Weekend Indicator)
This plot visualizes whether slow commute speeds correspond directly to fine particulate matter build-ups, and highlights the difference in weekend traffic workloads.

In [4]:
plt.figure(figsize=(10, 6))
scatter = sns.scatterplot(
    data=df,
    x="avg_speed_kmph",
    y="PM2.5",
    hue="is_weekend",
    palette={0: "#e74c3c", 1: "#2ecc71"},  # red for weekday, green for weekend
    alpha=0.85,
    s=80,
    edgecolor='w',
    linewidth=0.8
)

# Customizing aesthetics
plt.title("Bangalore Mobility vs. Air Quality: Speed vs. PM2.5 Concentrations", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("Average Travel Speed (km/h)", fontsize=11)
plt.ylabel("PM2.5 Concentration (ug/m3)", fontsize=11)
plt.legend(title="Is Weekend?", labels=["Weekday (0)", "Weekend (1)"], loc="upper right")
plt.tight_layout()
plt.show()

### Step 4: Calculate Pearson Correlation Coefficient (r) per Station
We iterate over all 5 individual municipal locations to calculate the exact Pearson correlation coefficient between local `congestion_level` and `AQI` readings.

In [5]:
print("=== Pearson Correlation Coefficients (r) per Station ===")
print("Evaluating localized traffic congestion level vs. environmental AQI:\n")

pearson_results = {}
for loc in df['location'].unique():
    loc_df = df[df['location'] == loc].dropna(subset=['congestion_level', 'AQI'])
    if len(loc_df) > 1:
        r_val = loc_df['congestion_level'].corr(loc_df['AQI'])
        pearson_results[loc] = r_val
        print(f"  Location: {loc:<15} | Pearson r = {r_val:.4f}")

### Step 5: Save and Compile Top 3 Insights

In [6]:
# Print the compiled insights report that was generated
insights_path = "../assets/insights.md"
if os.path.exists(insights_path):
    with open(insights_path, "r") as f:
        print(f.read())
else:
    print("Insights file was not found under /assets/insights.md.")